# Урок 3.1 — Spark с нуля: DataFrame и цикл read → inspect → transform → validate → write

Этот ноутбук — урок 3.1 модуля «Основы Spark».

В уроке работаем с полным датасетом заказов Olist:

- файл: `olist_orders_dataset.csv`
- путь внутри стенда: `/data/csv/olist_orders_dataset.csv`

---

## Что мы сделаем

1. Проверим `SparkSession`.
2. Прочитаем CSV из `/data/csv` в DataFrame.
3. Посмотрим данные и схему (`columns`, `printSchema()`, `show()`, `count()`).
4. Сделаем несколько простых трансформаций.
5. Сохраним результаты в `/workspace/lesson03_01/...` (CSV и Parquet).
6. Прочитаем результаты обратно и проверите себя через `count()`.

---

## Правило путей в стенде

- исходные данные читаем из `/data/csv`
- результаты уроков пишем в `/workspace`

Причина: и driver в Jupyter, и executors на воркерах должны видеть одинаковые пути.

---

## Важно про Jupyter

- Любую ячейку можно выполнять сколько угодно раз.
- Если повторно запускаем ячейку с `df = ...`, то переменная `df` просто **переприсваивается** (никаких “сломанных” DataFrame от этого не будет).


---
## 0. Импорты и SparkSession

В стенде `spark_01` Spark‑сессия часто уже доступна как переменная `spark`.
Если переменной нет — создадим её.

> Примеры:
> - `df.show()` — вызываем метод через точку  
> - `df.select(...).show()` — можно “цепочкой” вызвать несколько методов подряд


In [1]:
# импортируем то, что будем использовать в уроке.
# Если вы перезапустили ядро (Kernel) — просто выполните её снова.

import os
import shutil

from pyspark.sql import functions as F
from pyspark.sql import SparkSession


ModuleNotFoundError: No module named 'pyspark'

In [2]:
# проверяем, есть ли уже SparkSession в переменной `spark`.
# Если `spark` не существует — создаём SparkSession.

try:
    spark  # noqa: F821
except NameError:
    spark = SparkSession.builder.getOrCreate()


NameError: name 'SparkSession' is not defined

In [29]:
# просто выводим объект SparkSession, чтобы убедиться, что он создан.

spark


---
## 1. Пути

Определим ключевые пути урока.

- `/data/csv` — исходные CSV
- `/workspace` — рабочая зона
- `/workspace/lesson03_01` — результаты именно этого урока


In [3]:
# задаём пути и создаём папку урока (если её нет).

DATA_DIR = "/data/csv"
WORKSPACE_DIR = "/workspace"
LESSON_DIR = f"{WORKSPACE_DIR}/lesson03_01"

os.makedirs(LESSON_DIR, exist_ok=True)

ORDERS_CSV = f"{DATA_DIR}/olist_orders_dataset.csv"

# Куда будем писать результаты примеров из урока
OUT_ORDERS_FULL_CSV = f"{LESSON_DIR}/orders_full_csv"
OUT_ORDERS_FULL_PARQUET = f"{LESSON_DIR}/orders_full_parquet"

# Куда будем писать результаты практики (TASK01..TASK08)
TASK01_OUT = f"{LESSON_DIR}/task01_delivered_csv"
TASK02_OUT = f"{LESSON_DIR}/task02_unique_statuses_parquet"
TASK03_OUT = f"{LESSON_DIR}/task03_status_cnt_parquet"
TASK04_OUT = f"{LESSON_DIR}/task04_with_purchase_date_parquet"
TASK05_OUT = f"{LESSON_DIR}/task05_missing_delivery_csv"
TASK06_OUT = f"{LESSON_DIR}/task06_narrow_orders_csv"
TASK07_OUT = f"{LESSON_DIR}/task07_sorted_orders_parquet"
TASK08_OUT = f"{LESSON_DIR}/task08_status_share_parquet"


In [4]:
# быстро проверяем, что пути выглядят правильно.

(DATA_DIR, WORKSPACE_DIR, LESSON_DIR, ORDERS_CSV)


('/data/csv',
 '/workspace',
 '/workspace/lesson03_01',
 '/data/csv/olist_orders_dataset.csv')

In [5]:
# helper для “идемпотентной” записи.
# Идея: перед записью можно удалить папку результата и записать заново.

def rm_dir(path: str) -> None:
    shutil.rmtree(path, ignore_errors=True)


---
## 2. Читаем заказы из CSV: `header` и `inferSchema`

Когда читаем CSV в Spark, отвечаем на два вопроса:

1) Есть ли в файле строка с названиями колонок?  
2) Какие типы данных у колонок? (строка/число/дата и т.д.)


In [6]:
# читаем CSV в DataFrame.
# header=True -> первая строка CSV = названия колонок
# inferSchema=True -> Spark пытается угадать типы (удобно для обучения)

df_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_CSV)
)

df_orders


DataFrame[order_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp]

---
## 3. Первое знакомство с DataFrame

Посмотрим:

- список колонок
- схему (типы)
- несколько строк
- количество строк

> Важно: Spark ленивый. Пока вы не вызвали action (например, `show()` или `count()`), данные реально могут ещё не читаться полностью.


In [7]:
# смотрим список колонок (и их количество).

len(df_orders.columns), df_orders.columns


(8,
 ['order_id',
  'customer_id',
  'order_status',
  'order_purchase_timestamp',
  'order_approved_at',
  'order_delivered_carrier_date',
  'order_delivered_customer_date',
  'order_estimated_delivery_date'])

In [8]:
# смотрим схему (типы колонок).

df_orders.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [36]:
# смотрим первые 5 строк.
# truncate=False -> не обрезать длинные строки при выводе

df_orders.show(5, truncate=False)


+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |2017-10-02 11:07:15|2017-10-04 19:55:00         |2017-10-10 21:25:13          |2017-10-18 00:00:00          |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |2018-07-26 03:24:27|2018-07-26 14:3

In [9]:
# считаем количество строк.
# count() — это action (Spark реально выполнит работу)

df_orders.count()


99441

---
## 3.1 Небольшие проверки качества данных (простые)


In [ ]:
# показываем, как работает isNull() на примере одной колонки.

df_orders.select(
    "order_id",
    F.col("order_id").isNull().alias("is_order_id_null")
).show(5, truncate=False)


In [ ]:
# считаем количество NULL в ключевых полях.

df_orders.select(
    F.count("*").alias("rows"),
    F.sum(F.col("order_id").isNull().cast("int")).alias("null_order_id"),
    F.sum(F.col("customer_id").isNull().cast("int")).alias("null_customer_id"),
).show(truncate=False)


---
## 3.2 Дополнительно (можно пропустить в первый раз)

Ниже — две “внутренние” штуки Spark:

- сколько партиций у DataFrame
- как посмотреть план выполнения (`explain`).


In [ ]:
# смотрим количество партиций.

df_orders.rdd.getNumPartitions()


In [ ]:
# смотрим план выполнения для простого действия.

df_orders.select("order_id").explain("formatted")


---
## 4. Первая полезная трансформация

Соберём учебный датафрейм с базовыми полями и новой колонкой `order_purchase_date`.

Важно: мы **не меняем** исходный `df_orders`.  
Мы создаём новый DataFrame и записываем его в новую переменную.


In [ ]:
# выбираем только нужные колонки (select).

df_orders_basic = df_orders.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
)


In [ ]:
# добавляем новую колонку (withColumn).

df_orders_basic = df_orders_basic.withColumn(
    "order_purchase_date",
    F.to_date("order_purchase_timestamp")
)


In [ ]:
# проверяем результат.

df_orders_basic.show(5, truncate=False)


---
## 5. Мини‑сравнение: DataFrame API и Spark SQL

Один и тот же движок, два способа описать задачу.

Сделаем агрегацию: количество заказов по статусам.


In [ ]:
# считаем количество заказов по статусам через DataFrame API.

df_status_dfapi = (
    df_orders
    .groupBy("order_status")
    .agg(F.count("*").alias("cnt"))
    .orderBy(F.col("cnt").desc())
)

df_status_dfapi.show(truncate=False)


In [ ]:
# создаём temp view, чтобы писать SQL.

df_orders.createOrReplaceTempView("orders")


In [ ]:
# SQL-запрос как строка Python.

query = """
SELECT
    order_status,
    COUNT(*) AS cnt
FROM orders
GROUP BY order_status
ORDER BY cnt DESC
"""


In [ ]:
# выполняем SQL и смотрим результат.

df_status_sql = spark.sql(query)
df_status_sql.show(truncate=False)


---
## 6. Запись результатов урока в CSV и Parquet

Пишем в директорию.

- `.mode("overwrite")` — перезаписывает результат
- CSV обычно создаётся как набор файлов `part-...csv` в папке


In [ ]:
# чистим папку результата для CSV (на случай повторного запуска).

rm_dir(OUT_ORDERS_FULL_CSV)


In [ ]:
# записываем df_orders_basic в CSV с заголовком.

(
    df_orders_basic.write
    .mode("overwrite")
    .option("header", True)
    .csv(OUT_ORDERS_FULL_CSV)
)


In [ ]:
# смотрим файлы в папке CSV.

os.listdir(OUT_ORDERS_FULL_CSV)[:10]


In [ ]:
# чистим папку результата для Parquet.

rm_dir(OUT_ORDERS_FULL_PARQUET)


In [ ]:
# записываем df_orders_basic в Parquet.

(
    df_orders_basic.write
    .mode("overwrite")
    .parquet(OUT_ORDERS_FULL_PARQUET)
)


In [ ]:
# смотрим файлы в папке Parquet.

os.listdir(OUT_ORDERS_FULL_PARQUET)[:10]


---
## 7. Читаем файлы обратно и проверяем себя

Инженерная проверка по минимуму:

write → read → compare `count()`


In [ ]:
# читаем CSV обратно.

df_csv_back = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(OUT_ORDERS_FULL_CSV)
)


In [ ]:
# читаем Parquet обратно.

df_parquet_back = spark.read.parquet(OUT_ORDERS_FULL_PARQUET)


In [ ]:
# сравниваем count().

df_orders_basic.count(), df_csv_back.count(), df_parquet_back.count()


---
# Практика

Ниже — 8 заданий.

Общее правило:

- входные данные: DataFrame `df_orders` (из `/data/csv/olist_orders_dataset.csv`) или temp view `orders`
- для каждого задания:
  1) получите результирующий DataFrame
  2) сделайте `show(..., truncate=False)`
  3) запишите результат в указанную папку в `/workspace/lesson03_01/...`
  4) прочитайте результат обратно и сравните `count()`

Подсказка: чтобы записывать “идемпотентно”, можно вызывать `rm_dir(PATH)` перед `.write`.


## Задание 1. Заказы в статусе delivered

Входные данные:
- DataFrame `df_orders`

Что сделать:
- отфильтровать заказы со статусом `order_status = 'delivered'`

Что вывести:
- `order_id`, `customer_id`, `order_status`, `order_purchase_timestamp`

Куда записать результат:
- CSV (с заголовком) в `/workspace/lesson03_01/task01_delivered_csv`

Критерии приёмки:
- `show(5)`
- `count()` до/после записи совпадает после чтения обратно


In [ ]:
# читаем CSV в DataFrame.
df_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_CSV)
)

# фильтруем данные
df_status_filter = (
    df_orders
    .where(df_orders.order_status == 'delivered')
)

# выбираем нужные нам колонки
df_select_columns = df_status_filter.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp"
)

# проверяем результат
df_select_columns.show(5, truncate=False)

# считаем строки
# df_status_cnt = (
#     df_select_columns
#     .agg(F.count("*").alias("cnt"))
# )
# df_status_cnt.show(truncate=False)

# чистим папку результата для CSV (на случай повторного запуска).
rm_dir(TASK01_OUT)

# записываем df_status_filter в CSV с заголовком.
(
    df_select_columns.write
    .mode("overwrite")
    .option("header", True)
    .csv(TASK01_OUT)
)

# читаем CSV
df_csv_back = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(TASK01_OUT)
)

# проверяем результат
df_csv_back.show(5, truncate=False)

# сравниваем количество строк
print(f"status_filter: {df_select_columns.count()}, csv_back: {df_csv_back.count()}")


## Задание 2. Уникальные статусы заказов

Входные данные:
- DataFrame `df_orders`

Что сделать:
- получить список уникальных значений `order_status`

Что вывести:
- `order_status`

Сортировка:
- по `order_status` по возрастанию

Куда записать результат:
- Parquet в `/workspace/lesson03_01/task02_unique_statuses_parquet`

Критерии приёмки:
- `show(20)`
- `count()` после чтения обратно совпадает


In [ ]:
df_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_CSV)
)

df_status_filter = (
    df_orders
    .select("order_status")
    .distinct()
    .orderBy("order_status")
)

df_status_filter.show(20, truncate=False)

(
    df_status_filter.write
    .mode("overwrite")
    .option("header", True)
    .parquet(TASK02_OUT)
)

parquet_read = (
    spark.read
    .option("header", True)
    .option("interSchema", True)
    .parquet(TASK02_OUT)
)

print(f"df_orders_cnt: {df_status_filter.count()}, parquet_cnt: {parquet_read.count()}")

## Задание 3. Количество заказов по статусам двумя способами

Входные данные:
- DataFrame `df_orders`
- temp view `orders` (если используете SQL)

Что сделать:
- посчитать количество заказов в каждом `order_status`
- решить двумя способами:
  - DataFrame API
  - Spark SQL

Что вывести:
- `order_status`, `cnt`

Сортировка:
- по `cnt` по убыванию, затем по `order_status` по возрастанию

Куда записать результат:
- Parquet в `/workspace/lesson03_01/task03_status_cnt_parquet`

Критерии приёмки:
- результаты DF API и SQL дают одинаковый `count()` строк
- `count()` после чтения parquet совпадает


In [ ]:
# читаем csv
df_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_CSV)
)

# создаём temp view
df_orders.createOrReplaceTempView("orders")

query = """
SELECT
    order_status,
    COUNT(*) AS cnt
FROM orders
GROUP BY order_status
ORDER BY cnt DESC, order_status
"""

# выполняем sql и смотрим результат
print("SparkSQL")
df_status_sql = spark.sql(query)
df_status_sql.show(truncate=False)



print("DataFrame")
cnt_order_status = (
    df_orders
    .groupBy("order_status")
    .agg(F.count("*").alias("cnt"))
    .orderBy(F.col("cnt").desc())
)

cnt_order_status.show(truncate=False)

# чистим папку
rm_dir(TASK03_OUT)

(
    cnt_order_status.write
    .mode("overwrite")
    .option("header", True)
    .parquet(TASK03_OUT)
)

parquet_read = (
    spark.read
    .option("header", True)
    .option("inferShema", True)
    .parquet(TASK03_OUT)
)

print(f"Count strings in DataFrame: {cnt_order_status.count()}, count strings in parquet: {parquet_read.count()}")

## Задание 4. Добавить дату покупки без времени

Входные данные:
- DataFrame `df_orders`

Что сделать:
- добавить колонку `order_purchase_date` = дата из `order_purchase_timestamp` (без времени)

Что вывести:
- `order_id`, `order_purchase_timestamp`, `order_purchase_date`

Куда записать результат:
- Parquet в `/workspace/lesson03_01/task04_with_purchase_date_parquet`

Критерии приёмки:
- `order_purchase_date` имеет тип date
- `count()` после чтения parquet совпадает


In [ ]:
df_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_CSV)
)

df_select = (
    df_orders
    .select(
        "order_id",
        "order_purchase_timestamp"
    )
)

df_orders_new_col = df_select.withColumn(
    "order_purchase_date",
    F.to_date("order_purchase_timestamp")
)

df_orders_new_col.show(10, truncate=False)

rm_dir(TASK03_OUT)

(
    df_orders_new_col.write
    .mode("overwrite")
    .option("header", True)
    .parquet(TASK04_OUT)
)

df_read_parquet = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .parquet(TASK04_OUT)
)

print("Type columns in parquet:")
df_read_parquet.printSchema()

print(f"Count strings in DataFrame: {df_orders_new_col.count()}, count strings in parquet: {df_read_parquet.count()}")

## Задание 5. Заказы без даты доставки клиенту

Входные данные:
- DataFrame `df_orders`

Что сделать:
- отфильтровать строки, где `order_delivered_customer_date IS NULL`

Что вывести:
- `order_id`, `order_status`, `order_delivered_customer_date`

Куда записать результат:
- CSV (с заголовком) в `/workspace/lesson03_01/task05_missing_delivery_csv`

Критерии приёмки:
- `show(5)`
- `count()` после чтения csv совпадает


In [ ]:
df_orders = (
    spark.read
    .option("header", True)
    .option("inferShema", True)
    .csv(ORDERS_CSV)
)

df_filter = (
    df_orders
    .select(
        "order_id",
        "order_status",
        "order_delivered_customer_date"
    )
    .filter(F.col("order_delivered_customer_date").isNull())
)

df_filter.show(10, truncate=False)

(
    df_filter.write
    .mode("overwrite")
    .option("header", True)
    .option("inferSchema", True)
    .csv(TASK05_OUT)
)

csv_read = (
    spark.read
    .option("header", True)
    .option("infernalSchema", True)
    .csv(TASK05_OUT)
)

print(f"Count strings in DataFrame: {df_filter.count()}, count strings in csv: {csv_read.count()}")

## Задание 6. Узкая таблица заказов

Входные данные:
- DataFrame `df_orders`

Что сделать:
- оставить только колонки: `order_id`, `customer_id`, `order_status`

Что вывести:
- `order_id`, `customer_id`, `order_status`

Куда записать результат:
- CSV (с заголовком) в `/workspace/lesson03_01/task06_narrow_orders_csv`

Критерии приёмки:
- в результате ровно 3 колонки
- `count()` после чтения csv совпадает


In [ ]:
df_orders = (
    spark.read
    .option("header", True)
    .option("infernalSchema", True)
    .csv(ORDERS_CSV)
)

df_filter = (
    df_orders
    .select(
        "order_id",
        "customer_id",
        "order_status"
    )
)

df_filter.show(10, truncate=False)

(
    df_filter.write
    .mode("overwrite")
    .option("header", True)
    .option("inferSchema", True)
    .csv(TASK06_OUT)
)

df_read = (
    spark.read
    .option("header", True)
    .option("infernalSchema", True)
    .csv(TASK06_OUT)
)

print(f"Count strings in DataFrame: {df_filter.count()}, count strings in csv: {df_read.count()}")

## Задание 7. Сортировка по времени покупки

Входные данные:
- DataFrame `df_orders`

Что сделать:
- отсортировать заказы по `order_purchase_timestamp` по убыванию

Что вывести:
- `order_id`, `order_purchase_timestamp`, `order_status`

Куда записать результат:
- Parquet в `/workspace/lesson03_01/task07_sorted_orders_parquet`

Критерии приёмки:
- первая строка в результате имеет максимальный `order_purchase_timestamp`
- `count()` после чтения parquet совпадает


In [ ]:
df_orders = (
    spark.read
    .option("header", True)
    .option("infernalSchema", True)
    .csv(ORDERS_CSV)
)

df_select = (
    df_orders
    .select(
        "order_id",
        "order_purchase_timestamp",
        "order_status"
    )
    # .orderBy("order_purchase_timestamp", ascending=False)
    .orderBy(F.col("order_purchase_timestamp").desc())
)

df_select.show(10, truncate=False)

(
    df_select.write
    .mode("overwrite")
    .option("header", True)
    .parquet(TASK07_OUT)
)

df_read = (
    spark.read
    .option("header", True)
    .option("infernalSchema", True)
    .parquet(TASK07_OUT)
)

print(f"Count strings in DataFrame: {df_select.count()}, count strings in parquet: {df_read.count()}")

## Задание 8. Доля статуса от общего числа заказов

Входные данные:
- DataFrame `df_orders`

Что сделать:
- посчитать количество заказов по каждому `order_status`
- добавить колонку `share_pct` — доля статуса от общего числа заказов (в процентах)
- округлить `share_pct` до 2 знаков

Что вывести:
- `order_status`, `cnt`, `share_pct`

Сортировка:
- по `cnt` по убыванию, затем по `order_status` по возрастанию

Куда записать результат:
- Parquet в `/workspace/lesson03_01/task08_status_share_parquet`

Критерии приёмки:
- сумма `share_pct` по всем строкам близка к 100 (из-за округления возможна погрешность)
- `count()` после чтения parquet совпадает


In [ ]:
# from pyspark.sql.window import Window

df_orders = (
    spark.read
    .option("header", True)
    .option("infernalSchema", True)
    .csv(ORDERS_CSV)
)

df_select = (
    df_orders
    .groupBy("order_status")
    .agg(F.count("*").alias("cnt"))
    .withColumn("share_pct", F.round(F.col("cnt") / df_orders.count() * 100, 2))
    .orderBy(F.desc("cnt"), F.asc("order_status"))
)

df_select.show(truncate=False)

(
    df_select.write
    .mode("overwrite")
    .option("header", True)
    .parquet(TASK08_OUT)
)

parquet_read = (
    spark.read
    .option("header", True)
    .option("infernalSchema", True)
    .parquet(TASK08_OUT)
)

print(f"Count strings in DataFrame: {df_select.count()}, count strings in parquet: {parquet_read.count()}")

---
# ✅ Проверки

Запускайте check‑ячейки ниже: получите **OK** или **НЕ OK** + причину.

Если вы не делали практику — проверки будут падать (это нормально).


In [ ]:
# небольшие утилиты для вывода статуса проверок.

def ok(task: str, details: str = "") -> None:
    msg = f"OK — {task}"
    if details:
        msg += f" | {details}"
    print(msg)

def bad(task: str, err: Exception) -> None:
    print(f"НЕ OK — {task} | {err}")

def run_check(task: str, check_fn) -> None:
    try:
        check_fn()
        ok(task)
    except AssertionError as e:
        bad(task, e)


In [ ]:
# helper-функции для чтения результатов из папок.

def read_csv_dir(path: str):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )

def read_parquet_dir(path: str):
    return spark.read.parquet(path)

def ensure_view(df, view_name: str = "orders") -> None:
    df.createOrReplaceTempView(view_name)


## Проверка задания 1 — delivered (CSV)


In [ ]:
# проверяем TASK01.

def check_task01():
    expected_cnt = df_orders.filter(F.col("order_status") == "delivered").count()
    back = read_csv_dir(TASK01_OUT)

    assert back.count() == expected_cnt, "count() после read-back не совпал"
    assert back.columns == ["order_id", "customer_id", "order_status", "order_purchase_timestamp"], "неверный список/порядок колонок"
    assert back.filter(F.col("order_status") != "delivered").count() == 0, "в результате есть статусы кроме delivered"

run_check("TASK01", check_task01)


## Проверка задания 2 — уникальные статусы (Parquet)


In [ ]:
# проверяем TASK02.

def check_task02():
    back = read_parquet_dir(TASK02_OUT)

    assert back.columns == ["order_status"], "должна быть ровно 1 колонка order_status"
    assert back.count() == df_orders.select("order_status").distinct().count(), "count() уникальных статусов не совпал"

    rows = [r["order_status"] for r in back.collect()]
    assert rows == sorted(rows), "статусы должны быть отсортированы по возрастанию"

run_check("TASK02", check_task02)


## Проверка задания 3 — count по статусам (DF API + SQL) + запись Parquet


In [ ]:
# проверяем TASK03.

def check_task03():
    ensure_view(df_orders, "orders")

    df_dfapi_chk = (
        df_orders.groupBy("order_status")
        .agg(F.count("*").alias("cnt"))
        .orderBy(F.col("cnt").desc(), F.col("order_status").asc())
    )

    df_sql_chk = spark.sql("""
    SELECT order_status, COUNT(*) AS cnt
    FROM orders
    GROUP BY order_status
    ORDER BY cnt DESC, order_status ASC
    """)

    a = [(r["order_status"], int(r["cnt"])) for r in df_dfapi_chk.collect()]
    b = [(r["order_status"], int(r["cnt"])) for r in df_sql_chk.collect()]
    assert a == b, "DF API и SQL должны давать одинаковые строки (status, cnt)"

    back = read_parquet_dir(TASK03_OUT)
    assert back.count() == df_dfapi_chk.count(), "count() после read-back parquet не совпал"

run_check("TASK03", check_task03)


## Проверка задания 4 — order_purchase_date (Parquet)


In [ ]:
# проверяем TASK04.

def check_task04():
    back = read_parquet_dir(TASK04_OUT)

    assert back.count() == df_orders.count(), "count() должен совпадать с df_orders"
    assert "order_purchase_date" in back.columns, "нет колонки order_purchase_date"

    dtype = dict(back.dtypes).get("order_purchase_date")
    assert dtype == "date", f"order_purchase_date должен быть date, сейчас: {dtype}"

run_check("TASK04", check_task04)


## Проверка задания 5 — order_delivered_customer_date IS NULL (CSV)


In [ ]:
# проверяем TASK05.

def check_task05():
    expected_cnt = df_orders.filter(F.col("order_delivered_customer_date").isNull()).count()
    back = read_csv_dir(TASK05_OUT)

    assert back.count() == expected_cnt, "count() после read-back не совпал"
    assert back.filter(F.col("order_delivered_customer_date").isNotNull()).count() == 0, "есть строки, где delivery_date не NULL"

run_check("TASK05", check_task05)


## Проверка задания 6 — узкая таблица (CSV)


In [ ]:
# проверяем TASK06.

def check_task06():
    back = read_csv_dir(TASK06_OUT)
    assert back.columns == ["order_id", "customer_id", "order_status"], "должно быть ровно 3 колонки в этом порядке"
    assert back.count() == df_orders.count(), "count() должен совпадать с df_orders"

run_check("TASK06", check_task06)


## Проверка задания 7 — сортировка по времени покупки DESC (Parquet)


In [ ]:
# проверяем TASK07.
# Порядок строк в Parquet не гарантирован, поэтому явно сортируем перед .first().

def check_task07():
    back = read_parquet_dir(TASK07_OUT)

    assert back.count() == df_orders.count(), "count() должен совпадать с df_orders"

    max_ts = df_orders.agg(F.max("order_purchase_timestamp").alias("mx")).first()["mx"]

    first_ts = (
        back
        .orderBy(F.col("order_purchase_timestamp").desc())
        .select("order_purchase_timestamp")
        .first()["order_purchase_timestamp"]
    )

    assert first_ts == max_ts, "первая строка должна иметь максимальный order_purchase_timestamp"

run_check("TASK07", check_task07)


## Проверка задания 8 — доля статуса в % (Parquet)


In [ ]:
# проверяем TASK08.

def check_task08():
    back = read_parquet_dir(TASK08_OUT)

    assert back.columns == ["order_status", "cnt", "share_pct"], "ожидаются колонки: order_status, cnt, share_pct"
    assert back.count() == df_orders.select("order_status").distinct().count(), "кол-во строк должно равняться числу статусов"

    s = back.select(F.sum("share_pct").alias("s")).collect()[0]["s"]
    s = float(s) if s is not None else 0.0
    assert abs(s - 100.0) <= 0.5, f"сумма share_pct должна быть близка к 100, сейчас: {s}"

run_check("TASK08", check_task08)
